In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar
import contextily as ctx
import numpy as np
import textwrap

In [2]:
current_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Deliverables\Final Layers\Current_Road_Segment_Risk.shp"
future_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Deliverables\Final Layers\Future_Road_Segment_Risk.shp"
peru_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\GIS\5_ADMIN_BOUNDARIES_VIAS_ROADS_CIUDADES_CITIES\Admin_Boundary\Limite_Peru.shp"
roads_departmental = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\GIS\5_ADMIN_BOUNDARIES_VIAS_ROADS_CIUDADES_CITIES\Red Vial\Merged\Roads_Departmental.shp"
roads_national = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\GIS\5_ADMIN_BOUNDARIES_VIAS_ROADS_CIUDADES_CITIES\Red Vial\Merged\Roads_National.shp"
departmental_limits = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\GIS\5_ADMIN_BOUNDARIES_VIAS_ROADS_CIUDADES_CITIES\Limite_Demografico\Limite_Departamental.shp"
subbasins_shapefile = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\GIS_WaterALLOC_General\Peru_AHD_with_districts.shp"
output_file_current = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Deliverables\Final Layers\Road_Vulnerability_Current.png"
output_file_future = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Deliverables\Final Layers\Road_Vulnerability_Future_2050.png"


In [3]:
target_crs = "EPSG:3857"
peru_gdf = gpd.read_file(peru_shapefile).to_crs(target_crs)
current_segments = gpd.read_file(current_path).to_crs(target_crs)
future_segments = gpd.read_file(future_path).to_crs(target_crs)
roads_departmental = gpd.read_file(roads_departmental).to_crs(target_crs)
roads_national = gpd.read_file(roads_national).to_crs(target_crs)
departmental_limits = gpd.read_file(departmental_limits).to_crs(target_crs)
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs(target_crs)

In [4]:
risk_labels = ['Bajo', 'Medio', 'Alto']
risk_colors = ['green', 'yellow', 'red']

risk_color_map = dict(zip(risk_labels, risk_colors))

# map colors from Nivel
current_segments["plot_color"] = current_segments["Nivel"].map(risk_color_map)
future_segments["plot_color"] = future_segments["Nivel"].map(risk_color_map)

# keep segments with missing/unmatched values gray on the map
current_segments["plot_color"] = current_segments["plot_color"].fillna("lightgrey")
future_segments["plot_color"] = future_segments["plot_color"].fillna("lightgrey")


In [5]:
def add_north_arrow(ax, loc_x=0.92, loc_y=0.92, size=0.05):
    ax.annotate(
        "",
        xy=(loc_x, loc_y),
        xytext=(loc_x, loc_y - size),
        arrowprops=dict(
            facecolor="black",
            edgecolor="black",
            width=3,
            headwidth=10,
            headlength=10
        ),
        xycoords=ax.transAxes
    )

    ax.text(
        loc_x,
        loc_y + 0.015,
        "N",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        transform=ax.transAxes
    )

def add_scale_bar(ax, location="lower left"):
    scalebar = ScaleBar(
        dx=1,
        units="m",
        dimension="si-length",
        location=location,
        length_fraction=0.25,
        pad=0.15,
        box_alpha=0.6
    )
    ax.add_artist(scalebar)

def add_legend_row(ax_leg):
    ax_leg.axis("off")

    handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='None',
            markerfacecolor=color,
            markeredgecolor='black',
            markersize=14,
            label=label
        )
        for label, color in zip(risk_labels, risk_colors)
    ]

    ax_leg.legend(
        handles=handles,
        loc="center",
        ncol=3,
        frameon=False,
        handletextpad=0.8,
        columnspacing=2.0,
        fontsize=13,
    )


In [6]:
def plot_road_risk_single_scenario(roads_gdf, peru_gdf, roads_nat, roads_dep, title, output_file):
    fig = plt.figure(figsize=(10, 12))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[20, 2])

    ax = fig.add_subplot(gs[0, 0])
    ax_leg = fig.add_subplot(gs[1, 0])

    # Set map extent first
    xmin, ymin, xmax, ymax = peru_gdf.total_bounds
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Basemap first
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        crs=peru_gdf.crs,
        zoom=6,
        alpha=0.8
    )

    # Peru boundary only (no white fill, otherwise it hides basemap)
    peru_gdf.boundary.plot(
        ax=ax,
        edgecolor="black",
        linewidth=0.8
    )

    # Additional roads in gray
    roads_nat.plot(
        ax=ax,
        color="grey",
        linewidth=0.8,
        alpha=0.6
    )

    roads_dep.plot(
        ax=ax,
        color="grey",
        linewidth=0.6,
        alpha=0.6
    )

    # Vulnerable road segments
    roads_gdf.plot(
        ax=ax,
        color=roads_gdf["plot_color"],
        linewidth=2.5
    )

    ax.set_title(title, fontsize=13)
    ax.set_axis_off()

    add_north_arrow(ax)
    add_scale_bar(ax, location="lower left")
    add_legend_row(ax_leg)

    fig.subplots_adjust(top=0.95, bottom=0.06, hspace=0.02)

    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close()

In [7]:
plot_road_risk_single_scenario(
    roads_gdf=current_segments,
    peru_gdf=peru_gdf,
    roads_nat=roads_national,
    roads_dep=roads_departmental,
    title="Riesgo de Segmentos de Carretera a Nivel Nacional: Escenario Actual",
    output_file=output_file_current
)

plot_road_risk_single_scenario(
    roads_gdf=future_segments,
    peru_gdf=peru_gdf,
    roads_nat=roads_national,
    roads_dep=roads_departmental,
    title="Riesgo de Segmentos de Carretera a Nivel Nacional: Escenario Futuro al 2050",
    output_file=output_file_future
)

In [8]:
summary_output_dir = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2c_AnalisisBlueSpot\Deliverables\Final Layers"

risk_order_plot = ["Alto", "Medio", "Bajo"]
risk_colors_plot = {
    "Bajo": "green",
    "Medio": "yellow",
    "Alto": "red"
}

scenario_dict = {
    "actual": {
        "gdf": current_segments.copy(),
        "tag": "Actual"
    },
    "future_2050": {
        "gdf": future_segments.copy(),
        "tag": "Futuro al 2050"
    }
}


def prepare_polygon_areas(gdf, name_col):
    gdf = gdf[[name_col, "geometry"]].copy()
    gdf = gdf.dropna(subset=[name_col]).copy()
    gdf["area_km2"] = gdf.geometry.area / 1e6

    # aggregate to one row per polygon name
    gdf = (
        gdf.groupby(name_col, as_index=False)["area_km2"]
        .sum()
    )

    return gdf


def assign_segments_to_polygons_by_representative_point(segments_gdf, polygons_gdf, polygon_name_col):
    """
    Assign each road segment to one polygon using a representative point.
    This avoids double-counting lines that cross boundaries.
    """
    seg = segments_gdf.copy()
    seg = seg.reset_index(drop=True)
    seg["segment_id"] = seg.index.astype(int)
    seg["rep_point"] = seg.geometry.representative_point()

    seg_points = gpd.GeoDataFrame(
        seg.drop(columns="geometry"),
        geometry=seg["rep_point"],
        crs=seg.crs
    )

    joined = gpd.sjoin(
        seg_points,
        polygons_gdf[[polygon_name_col, "geometry"]],
        how="left",
        predicate="within"
    )

    joined = joined.drop(columns=["rep_point"], errors="ignore")
    return joined


def summarize_national_counts(segments_gdf):
    summary = (
        segments_gdf.groupby("Nivel")
        .size()
        .reindex(["Bajo", "Medio", "Alto"], fill_value=0)
        .reset_index(name="count")
    )

    summary["percent"] = 100 * summary["count"] / summary["count"].sum()
    return summary


def summarize_by_polygon_normalized(segments_with_polygon, polygons_gdf, polygon_name_col):
    counts = (
        segments_with_polygon.dropna(subset=[polygon_name_col])
        .groupby([polygon_name_col, "Nivel"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["Alto", "Medio", "Bajo"], fill_value=0)
        .reset_index()
    )

    poly_areas = prepare_polygon_areas(polygons_gdf, polygon_name_col)

    summary = counts.merge(
        poly_areas[[polygon_name_col, "area_km2"]],
        on=polygon_name_col,
        how="left"
    )

    for col in ["Alto", "Medio", "Bajo"]:
        summary[f"{col}_norm"] = summary[col] / summary["area_km2"]

    summary["total"] = summary[["Alto", "Medio", "Bajo"]].sum(axis=1)
    summary["total_norm"] = summary[[f"{c}_norm" for c in ["Alto", "Medio", "Bajo"]]].sum(axis=1)

    return summary


def plot_national_risk_categories(summary_df, scenario_tag, output_file):
    fig, ax = plt.subplots(figsize=(9, 6.5))

    x_labels = [
        "Riesgo bajo",
        "Riesgo medio",
        "Riesgo alto"
    ]

    counts = summary_df.set_index("Nivel").loc[["Bajo", "Medio", "Alto"], "count"].values
    percents = summary_df.set_index("Nivel").loc[["Bajo", "Medio", "Alto"], "percent"].values
    colors = [risk_colors_plot["Bajo"], risk_colors_plot["Medio"], risk_colors_plot["Alto"]]

    bars = ax.bar(x_labels, counts, color=colors, width=0.32)

    ymax = counts.max() * 1.22
    ax.set_ylim(0, ymax)

    for bar, count, pct in zip(bars, counts, percents):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + ymax * 0.03,
            f"{count:,.0f}\n{pct:.0f}%",
            ha="center",
            va="bottom",
            fontsize=11
        )

    ax.set_title(
        f"Segmentos de Carretera por Categoría de Riesgo\nEscenario {scenario_tag}",
        fontsize=20,
        pad=14
    )
    ax.set_ylabel("Número de Segmentos de Carretera", fontsize=16)
    ax.grid(axis="x", color="#c9c9c9")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close()


def plot_top_polygons_stacked(summary_df, name_col, scenario_tag, output_file, title_main):
    df = summary_df.copy()

    # keep all polygons with at least one segment
    df = df[df["total"] > 0].copy()

    # sort by normalized total risk density
    df = df.sort_values("total_norm", ascending=False).copy()

    x = np.arange(len(df))
    alto = df["Alto_norm"].values
    medio = df["Medio_norm"].values
    bajo = df["Bajo_norm"].values

    fig, ax = plt.subplots(figsize=(16, 8))

    ax.bar(x, alto, color=risk_colors_plot["Alto"], width=0.55, label="Riesgo alto")
    ax.bar(x, medio, bottom=alto, color=risk_colors_plot["Medio"], width=0.55, label="Riesgo medio")
    ax.bar(x, bajo, bottom=alto + medio, color=risk_colors_plot["Bajo"], width=0.55, label="Riesgo bajo")

    if name_col == "Cuenca":
        labels_clean = [str(lbl).replace("Cuenca ", "") for lbl in df[name_col]]
        wrap_width = 7
    else:
        labels_clean = [str(lbl) for lbl in df[name_col]]
        wrap_width = 12

    wrapped_labels = [
        "\n".join(textwrap.wrap(lbl, width=wrap_width))
        for lbl in labels_clean
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(
    wrapped_labels,
    rotation=35,
    ha="right",
    fontsize=8,
    linespacing=1.2
    )

    ax.set_ylabel("Segmentos de Carretera por km²", fontsize=14)
    ax.set_title(f"{title_main}\nEscenario {scenario_tag}", fontsize=18, pad=14)

    ax.grid(axis="y", color="#c9c9c9")
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        frameon=False,
        fontsize=12
    )

    plt.subplots_adjust(bottom=0.38, top=0.88)
    plt.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close()


for scenario_key, scenario_info in scenario_dict.items():

    roads_gdf = scenario_info["gdf"].copy()
    scenario_tag = scenario_info["tag"]

    # --- National summary ---
    national_summary = summarize_national_counts(roads_gdf)

    plot_national_risk_categories(
        national_summary,
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Category_Summary_{scenario_key}.png"
    )

    # --- By department (normalized by department area) ---
    dept_join = assign_segments_to_polygons_by_representative_point(
        roads_gdf,
        departmental_limits,
        "NOM_DEP"
    )

    dept_summary = summarize_by_polygon_normalized(
        dept_join,
        departmental_limits,
        "NOM_DEP"
    )

    plot_top_polygons_stacked(
        dept_summary,
        name_col="NOM_DEP",
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Departments_{scenario_key}.png",
        title_main="Departamentos con más Segmentos de Carretera en Riesgo"
    )

    # --- By cuenca (normalized by cuenca area) ---
    cuenca_join = assign_segments_to_polygons_by_representative_point(
        roads_gdf,
        subbasins_gdf,
        "Cuenca"
    )

    cuenca_summary = summarize_by_polygon_normalized(
        cuenca_join,
        subbasins_gdf,
        "Cuenca"
    )

    plot_top_polygons_stacked(
        cuenca_summary,
        name_col="Cuenca",
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Cuencas_{scenario_key}.png",
        title_main="Cuencas con más Segmentos de Carretera en Riesgo"
    )

In [9]:
for scenario_key, scenario_info in scenario_dict.items():

    roads_gdf = scenario_info["gdf"].copy()
    scenario_tag = scenario_info["tag"]

    # --- National summary ---
    national_summary = summarize_national_counts(roads_gdf)

    plot_national_risk_categories(
        national_summary,
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Category_Summary_{scenario_key}.png"
    )

    # --- By department (normalize by area using departmental shapefile) ---
    dept_join = assign_segments_to_polygons_by_representative_point(
        roads_gdf,
        departmental_limits,
        "NOM_DEP"
    )

    dept_summary = summarize_by_polygon_normalized(
        dept_join,
        departmental_limits,
        "NOM_DEP"
    )

    plot_top_polygons_stacked(
        dept_summary,
        name_col="NOM_DEP",
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Departments_{scenario_key}.png",
        title_main="Departamentos con más Segmentos de Carretera en Riesgo",
    )

    # --- By cuenca (normalize by area using subbasins shapefile) ---
    cuenca_join = assign_segments_to_polygons_by_representative_point(
        roads_gdf,
        subbasins_gdf,
        "Cuenca"
    )

    cuenca_summary = summarize_by_polygon_normalized(
        cuenca_join,
        subbasins_gdf,
        "Cuenca"
    )

    plot_top_polygons_stacked(
        cuenca_summary,
        name_col="Cuenca",
        scenario_tag=scenario_tag,
        output_file=fr"{summary_output_dir}\Road_Risk_Cuencas_{scenario_key}.png",
        title_main="Cuencas con más Segmentos de Carretera en Riesgo",
    )